In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import sys
sys.path.append('..')
from modules.source import OUTPUT_FOLDER_PATH, parquet_daily, parquet_monthly, Sources
from modules.helpers import add_regime_shading

df_monthly = pd.read_parquet(parquet_monthly)
df_daily = pd.read_parquet(parquet_daily)

plt.rcParams.update({ # set default settings
   "figure.dpi": 150, "font.size": 8, "font.family": "sans-serif",
   "axes.grid": True, "grid.linestyle": "--", "grid.alpha": 0.4,
   "axes.titlesize": 9, "axes.titleweight": "bold", "axes.labelweight": "bold"
})

DISPLAY_NAMES = {
   Sources.rate_differential: "FFR-OPR Spread",
   Sources.KLCI: "KLCI",
   Sources.crude_palm_oil: "Crude Palm Oil",
   Sources.brent_oil: "Brent Oil",
   Sources.UST_10Y: "UST 10Y Yield",
   Sources.VIX: "VIX",
   Sources.DXY: "DXY",
   Sources.cpi_inflation: "MY CPI Inflation Index",
}

In [ ]:
variables = [Sources.KLCI, Sources.crude_palm_oil, Sources.cpi_inflation, Sources.rate_differential, Sources.brent_oil, Sources.UST_10Y, Sources.VIX, Sources.DXY]

fig, axes = plt.subplots(len(variables), 2, figsize=(10, 1.9 * len(variables)), sharex=True, layout="constrained")

axes[0, 0].set_title("Level")
axes[0, 1].set_title(r"Change ($\Delta$ / Log Return)")
axes[0, 1].scatter([], [], color="#dc2626", s=15, label="outliers")
axes[0, 1].legend(fontsize=6, loc="upper left", frameon=False)

for (ax_lvl, ax_chg), v in zip(axes, variables):
   level, change = df_monthly[(v, "price")], df_monthly[(v, "change")]

   ax_lvl.plot(level, lw=1)
   ax_lvl.set_ylabel(DISPLAY_NAMES.get(v, v))

   ax_chg.plot(change, color="#374151", lw=0.8)
   ax_chg.axhline(color="#9ca3af", lw=0.4)

   abs_z = ((change - change.mean()) / change.std()).abs()
   outlier_date = abs_z[abs_z > 2.5].nlargest(3).index
   ax_chg.scatter(outlier_date, change.loc[outlier_date], color="#dc2626", s=15, zorder=3)

fig.suptitle("Variance Question Raw Level and Stationary Variables Charts", fontsize=14, fontweight="bold")
fig.savefig(f"{OUTPUT_FOLDER_PATH}/variance_question_charts.png", bbox_inches="tight")